# CasADi Models – README Examples

Code snippets from the `casadi-models` README, verified to run.

## Building Models

In [ ]:
from cas_models.continuous_time.models import (
    StateSpaceModelCT, SSModelCTLinearFOSISO, SSModelCTLinearO2NoGainSISO
)
from cas_models.transformations import connect_systems_in_series

In [ ]:
# First order, single-input, single-output, continuous-time
# state-space model with symbolic parameters
sys_model = SSModelCTLinearFOSISO()
print(sys_model.f)
print(sys_model.h)

In [ ]:
# First order SISO system with fixed gain
sys_model = SSModelCTLinearFOSISO(K=2.5)
print(sys_model.f)
print(sys_model.h)

In [ ]:
# Second-order system with gain = 1
sys_model_2 = SSModelCTLinearO2NoGainSISO()
print(sys_model_2.f)
print(sys_model_2.h)

In [ ]:
# Combine both systems by connecting in series
sys = connect_systems_in_series([sys_model, sys_model_2], model_class=StateSpaceModelCT)
print(sys.f)
print(sys.h)

In [ ]:
# Series connections can also be made with the '*' operator
sys = sys_model * sys_model_2

## Simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cas_models.continuous_time.models import SSModelCTLinearFOSISO
from cas_models.discrete_time.models import StateSpaceModelDTFromCT
from cas_models.discrete_time.simulate import make_n_step_simulation_function_from_model

In [ ]:
# Continuous time model
sys_ct = SSModelCTLinearFOSISO(K=1, T1=2)

# Convert to discrete-time with dt=0.1
dt = 0.1
sys_dt = StateSpaceModelDTFromCT(sys_ct, dt)
print(sys_dt.F)
print(sys_dt.H)

In [ ]:
# Number of time-steps to simulate
nT = 100
simulate = make_n_step_simulation_function_from_model(sys_dt, nT)
print(simulate)

In [ ]:
# Simulation time
# Note: Simulation outputs include values for nT+1 time instants
t = dt * np.arange(nT + 1)
t_in = t[:-1]

# Simulation inputs
U = np.zeros((nT, 1))
U[t_in >= 1] = 1.0

# Initial condition
x0 = np.zeros(sys_dt.n)
X, Y = simulate(t, U, x0)

assert X.shape == (nT + 1, sys_dt.n)  # states
assert Y.shape == (nT + 1, sys_dt.ny)  # outputs

In [ ]:
# Make time-series plot
fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot(t, Y[:, 0])
ax.set_xlabel("t")
ax.set_ylabel("y")
ax.grid(False)
plt.tight_layout()
plt.show()

## Simulating a Feedback Loop

In [ ]:
from cas_models.continuous_time.regulators import SSModelCTPIInt
from cas_models.transformations import connect_feedback_system

# First-order plant: G(s) = 1 / (2s + 1)
plant = SSModelCTLinearFOSISO(K=1, T1=2, name="plant")
plant.describe()

In [ ]:
# PI controller in interactive form: Gc(s) = Kc * (Ti*s + 1) / (Ti*s)
ctrl = SSModelCTPIInt(name="ctrl")
ctrl.describe()

In [ ]:
# Negative feedback loop: reference -> ctrl -> plant -> (feedback) -> ctrl
sys1 = ctrl * plant
sys1.describe()

In [ ]:
sys_cl = connect_feedback_system(sys1, model_class=StateSpaceModelCT)
sys_cl.describe()

In [ ]:
# Convert to discrete-time and simulate step response
dt = 0.1
sys_cl_dt = StateSpaceModelDTFromCT(sys_cl, dt)

# Create CasADi simulation function
nT = 200
simulate_cl = make_n_step_simulation_function_from_model(sys_cl_dt, nT)
simulate_cl

In [ ]:
t = dt * np.arange(nT + 1)
R_full = np.where(t >= 1, 1.0, 0.0)      # reference at all nT+1 time instants
R = R_full[:-1].reshape(-1, 1)           # inputs at nT steps

# Set parameters for simulation
Ti = 2
Kc = 1
x0 = np.zeros(sys_cl_dt.n)
X, Y = simulate_cl(t, R, x0, Kc, Ti)

assert X.shape == (nT + 1, sys_cl_dt.n)
assert Y.shape == (nT + 1, sys_cl_dt.ny)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 2.5))
ax.step(t, R_full, "k--", where="post", label="reference")
ax.plot(t, Y[:, 0], label="output y")
ax.set_xlabel("t")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()